# DFL TranSPORTmer · 공 좌표 복원과 데드볼

- **설정** · 128차원, 16헤드, 최대 60 epoch, 9.6초 입력. 기존 두 단계 시간·선수 간 attention을 확장한다.
- **입력과 정답** · 공 토큰의 좌표는 항상 가린다. 실제로 관측된 인플레이 공 좌표로 복원 손실을 계산하고, `BallStatus`로 데드볼 이진 분류를 학습한다.
- **검증** · DFL 7경기 중 하나씩 바깥 평가에 둔다. 학습용 6경기 각 하프의 중앙 12% 창은 조기 종료용이다. 각 창은 겹치지 않는다. 조기 종료는 비터비 적용 후 내부 인플레이 구간 F1로 결정한다.
- **실행** · 같은 폴더의 `dfl_transportmer.ipynb`와 동일한 PyTorch/SciPy 환경. `MTAH_TRANSPORTMER_MODE=benchmark`는 속도 측정이다. 일반 실행은 7경기를 평가한다. 조기 종료용 가중치는 메모리에 유지한다.
- **출력** · `model/영상추출/추출csv/dead_ball/dfl_transportmer_joint_20261112_01/`.
- [TranSPORTmer 논문](https://openaccess.thecvf.com/content/ACCV2024/papers/Capellera_TranSPORTmer_A_Holistic_Approach_to_Trajectory_Understanding_in_Multi-Agent_Sports_ACCV_2024_paper.pdf)

In [ ]:
import json
from pathlib import Path

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/train_dfl.py").is_file())
folder = root / "model/영상추출/모델/dead_ball"
original = folder / "dfl_transportmer.ipynb"
original_cells = {cell["id"]: cell for cell in json.loads(original.read_text())["cells"]}
for index in (2, 4, 6, 8):
    cell = original_cells[f"dfl-transportmer-{index}"]
    exec(compile("".join(cell["source"]), f"{original}:{index}", "exec"))

WIDTH, HEADS, EPOCHS = 128, 16, 60
BATCH = 16
PATIENCE = 5
MIN_DELTA = 0.001
MIN_EPOCHS = 6
INNER_SHARE = 0.12
STATE_WEIGHT = 4.0
AMP = os.environ.get("MTAH_TRANSPORTMER_AMP", "fp32")
assert AMP in ("fp32", "bf16")
if AMP == "bf16":
    assert DEVICE.type == "mps", DEVICE
RUN_ID = "dfl_transportmer_joint_20261112_01"
OUT = ROOT / "model/영상추출/추출csv/dead_ball" / RUN_ID
NOTEBOOK = folder / "dfl_transportmer_joint.ipynb"

## 공 토큰과 두 출력

공 입력은 학습된 토큰으로 대체한다. 좌표 헤드는 피치 크기로 정규화한 좌표에서 평균 유클리드 거리를 최소화한다. 상태 헤드는 이진 교차 엔트로피를 계산한다. 합산 손실은 좌표 손실 + 4 × 상태 손실이다.

In [ ]:
class JointTransportmer(TransportmerDeadBall):
    def __init__(self):
        super().__init__()
        self.ball = nn.Parameter(torch.zeros(1, 1, 1, WIDTH))
        self.ball_head = nn.Sequential(nn.LayerNorm(WIDTH), nn.Linear(WIDTH, 2))
        nn.init.normal_(self.ball, std=0.02)
        nn.init.normal_(self.cls, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, players, valid):
        batch, steps, _, _ = players.shape
        observed = (players[..., 3] > 0) & valid[:, :, None]
        agents = observed.any(dim=1)
        tokens = torch.cat((self.embedding(players), self.ball.expand(batch, steps, 1, WIDTH),
                            self.cls.expand(batch, steps, 1, WIDTH)), dim=2)
        coarse = torch.cat((observed, valid[:, :, None].expand(batch, steps, 2)), dim=2).transpose(1, 2)
        coarse = coarse | (~coarse.any(-1, keepdim=True) & self.first)
        social = torch.cat((agents, torch.ones((batch, 2), dtype=torch.bool, device=players.device)), dim=1)
        fine = valid[:, None, :].expand_as(coarse)
        tokens = self.coarse(tokens + self.positional, coarse, social)
        tokens = self.fine(tokens + self.positional, fine, social)
        return self.classifier(tokens[:, :, -1]).squeeze(-1), self.ball_head(tokens[:, :, -2])


def prepare_joint(match_id, full=True):
    tracking, events, meta = dfl_load.load_match(match_id)
    match = features.match_from_dfl(tracking, meta)
    label = dfl_labels.dead_label(tracking)
    seed = CONFIG["sim_seed"] + MATCHES.index(match_id)
    noisy = simulate.simulate(match, seed=seed, **train_dfl.random_settings(np.random.default_rng(seed)))
    training = pack_windows(noisy, label, TRAIN_HOP)
    normalized_ball = match.ball / np.asarray(match.pitch, dtype=np.float32) * 2 - 1
    ball = normalized_ball[training["indices"]]
    observed_in_play = np.isfinite(ball).all(axis=-1) & (training["target"] == 0)
    training.update(
        ball=np.where(observed_in_play[..., None], ball, 0).astype(np.float32),
        ball_target=observed_in_play, label=label, period=np.asarray(match.period),
    )
    if not full:
        return training, None
    evaluation = pack_windows(simulate.simulate(match, seed=CONFIG["sim_seed"]), label, VALID_HOP)
    evaluation.update(label=label, period=np.asarray(match.period), source_frame=np.asarray(match.frame),
                      spells=dfl_labels.dead_spells(tracking, events), geometry=final_dead_balls.ball_signals(match),
                      pitch=match.pitch, ball_m=match.ball)
    evaluation["crossings"] = final_dead_balls.line_crossings(evaluation["geometry"])
    return training, evaluation


def internal_split(data):
    periods = data["period"][data["indices"][:, 0]]
    groups = []
    for half in np.unique(periods):
        indices = np.flatnonzero(periods == half)
        length = max(1, round(len(indices) * INNER_SHARE))
        start = (len(indices) - length) // 2
        hold = indices[start:start + length]
        assert data["valid"][hold].all()
        groups.append(hold)
    validation = np.concatenate(groups)
    training = np.setdiff1d(np.arange(len(periods)), validation, assume_unique=True)
    assert not np.intersect1d(data["indices"][training][data["valid"][training]],
                             data["indices"][validation][data["valid"][validation]]).size
    return training, groups


def amp():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP == "bf16")


def new_joint():
    torch.manual_seed(SEED)
    model = JointTransportmer().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    return model, opt


def train_joint_batch(model, opt, data, indices):
    x, valid, label = move(data, indices, include_target=True)
    ball = torch.from_numpy(data["ball"][indices]).to(DEVICE)
    ball_mask = torch.from_numpy(data["ball_target"][indices]).to(DEVICE)
    opt.zero_grad(set_to_none=True)
    with amp():
        logits, position = model(x, valid)
    bce = (F.binary_cross_entropy_with_logits(logits.float(), label.clamp_min(0), reduction="none") * valid).sum() / valid.sum()
    ade = (torch.linalg.vector_norm(position.float() - ball, dim=-1) * ball_mask).sum() / ball_mask.sum().clamp_min(1)
    loss = ade + STATE_WEIGHT * bce
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 5.0, foreach=False)
    opt.step()
    return torch.stack((loss.detach(), bce.detach(), ade.detach()))


def model_predictions(model, data, indices):
    scores, balls = [], []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(indices), BATCH):
            x, valid = move(data, indices[start:start + BATCH])
            with amp():
                logits, coordinates = model(x, valid)
            scores.append(logits.float().sigmoid().cpu().numpy())
            balls.append(coordinates.float().cpu().numpy())
    return np.concatenate(scores), np.concatenate(balls)


def inner_f1(model, datasets, splits, matches):
    f1 = []
    for match in matches:
        data = datasets[match]
        labels, periods, predictions = [], [], []
        for half_index, group in enumerate(splits[match][1]):
            clip, _ = model_predictions(model, data, group)
            anchors = data["indices"][group].ravel()
            assert (np.diff(anchors) == STEP).all()
            rows = np.arange(anchors[0], anchors[-1] + STEP)
            labels.append(data["label"][rows])
            periods.append(np.full(len(rows), half_index, dtype=np.int8))
            predictions.append(np.interp(rows, anchors, clip.ravel()))
        period, label, probability = map(np.concatenate, (periods, labels, predictions))
        state = smoothing.viterbi(probability, period, VITERBI["penalty"], VITERBI["bias"])
        f1.append(compare_smoothing.score_one(state, pd.DataFrame(dict(dead=label, period=period)))["alive_spell_f1"])
    return float(np.mean(f1))


def check_joint():
    model, _ = new_joint()
    model.eval()
    with torch.inference_mode():
        data = torch.zeros((2, WINDOW, SLOTS, 4), device=DEVICE)
        data[0, :, :4, :3] = torch.randn(WINDOW, 4, 3, device=DEVICE)
        data[0, :, :4, 3] = 1
        valid = torch.ones((2, WINDOW), dtype=torch.bool, device=DEVICE)
        result = model(data, valid)
        reordered = model(data.flip(dims=(2,)), valid)
    for a, b in zip(result, reordered):
        assert torch.isfinite(a).all()
        torch.testing.assert_close(a.cpu(), b.cpu(), atol=2e-4, rtol=2e-4)
    assert result[0].shape == (2, WINDOW) and result[1].shape == (2, WINDOW, 2)
    return sum(parameter.numel() for parameter in model.parameters())


def benchmark_joint():
    data, _ = prepare_joint(MATCHES[0], full=False)
    rng = np.random.default_rng(SEED)
    samples = [rng.choice(len(data["x"]), BATCH, replace=False) for _ in range(20)]
    model, opt = new_joint()
    for indices in samples[:5]:
        train_joint_batch(model, opt, data, indices)
    synchronize()
    start = perf_counter()
    for indices in samples[5:]:
        losses = train_joint_batch(model, opt, data, indices)
    synchronize()
    assert torch.isfinite(losses).all().item()
    print(f"{DEVICE} {AMP} | {(perf_counter()-start)/15:.4f}s per training batch", flush=True)

## 조기 종료

6 epoch부터 모니터링을 시작하고 5회 연속 0.001 이상 개선되지 않으면 중단한다. 최고 내부 점수의 가중치로 바깥 경기를 평가한다. 최고 가중치는 메모리에만 둔다.

In [ ]:
def cpu_weights(model):
    return {key: tensor.detach().cpu().clone() for key, tensor in model.state_dict().items()}


def save_json(value, path):
    pending = path.with_suffix(".tmp")
    pending.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    pending.replace(path)


def fit_joint(held, matches, datasets, splits):
    model, opt = new_joint()
    fields = ("x", "valid", "target", "ball", "ball_target")
    train = {key: np.concatenate([datasets[m][key][splits[m][0]] for m in matches]) for key in fields}
    history = []
    best_score, best_epoch, best, stale = -np.inf, 0, None, 0
    for epoch in range(1, EPOCHS + 1):
        started = perf_counter()
        model.train()
        order = np.random.default_rng(SEED + epoch).permutation(len(train["x"]))
        losses = [train_joint_batch(model, opt, train, order[i:i+BATCH]) for i in range(0, len(order), BATCH)]
        loss, bce, ade = map(float, torch.stack(losses).mean(0).cpu().tolist())
        assert all(np.isfinite((loss, bce, ade)))
        validation = inner_f1(model, datasets, splits, matches) if epoch >= MIN_EPOCHS else None
        if validation is not None:
            if validation > best_score + MIN_DELTA:
                best_score, best_epoch, best, stale = validation, epoch, cpu_weights(model), 0
            else:
                stale += 1
        seconds = perf_counter() - started
        history.append(dict(epoch=epoch, seconds=seconds))
        state = "대기" if validation is None else f"{validation:.4f}"
        print(f"  {held} epoch {epoch:02}/{EPOCHS} | {seconds:.1f}s | BCE {bce:.4f} | "
              f"공 손실 {ade:.4f} | 내부 구간 F1 {state} | best {best_epoch} | patience {stale}/{PATIENCE}", flush=True)
        if stale >= PATIENCE:
            break
    assert best is not None and best_epoch >= MIN_EPOCHS
    model.load_state_dict(best)
    return model.eval(), history, best_epoch, best_score


def merge_predictions(model, evaluation):
    probs, coords = model_predictions(model, evaluation, np.arange(len(evaluation["x"])))
    valid = evaluation["valid"]
    indices = evaluation["indices"][valid]
    blend = 0.25 + np.sin(np.pi * (np.arange(WINDOW) + 0.5) / WINDOW) ** 2
    weights = np.broadcast_to(blend, valid.shape)[valid]
    n = len(evaluation["label"])
    sums = np.zeros((n, 3), dtype=np.float64)
    counts = np.zeros(n, dtype=np.float64)
    np.add.at(sums, indices, np.column_stack((probs[valid], coords[valid])) * weights[:, None])
    np.add.at(counts, indices, weights)
    sampled = counts > 0
    full = np.empty_like(sums)
    for half in halves(evaluation["period"]):
        rows = np.arange(half.start, half.stop)
        at = rows[sampled[half]]
        np.testing.assert_array_equal(at, rows[::STEP])
        for column in range(full.shape[1]):
            full[half, column] = np.interp(rows, at, sums[at, column] / counts[at])
    assert np.isfinite(full).all() and np.all((0 <= full[:, 0]) & (full[:, 0] <= 1))
    return full[:, 0], (full[:, 1:] + 1) * 0.5 * evaluation["pitch"], sampled


def score_joint(evaluation, prob):
    scores, viterbi, forced = score(evaluation, prob)
    label = evaluation["label"]
    period = evaluation["period"]
    for row, state in zip(scores, (viterbi, forced)):
        frame_metrics = metrics.frame_metrics(label, state)
        row.update({key: float(frame_metrics[key]) for key in ("dead_precision", "dead_recall", "frame_acc")})
    raw = (prob >= 0.5).astype(np.int8)
    raw_row = compare_smoothing.score_one(raw, pd.DataFrame(dict(dead=label, period=period)))
    raw_metrics = metrics.frame_metrics(label, raw)
    raw_row.update(stage="raw", **{key: float(raw_metrics[key]) for key in ("dead_precision", "dead_recall", "frame_acc")})
    return [raw_row, *scores], viterbi, forced

## 결과 저장

보고 지표는 데드볼 프레임 F1·정밀도·재현율과 인플레이 구간 F1이다. 공 좌표 ADE는 실제 관측된 인플레이 프레임에서만 계산한다. 바깥 경기 예측과 점수는 별도 실행 폴더에 저장한다.

In [ ]:
def run_joint(parameters):
    OUT.mkdir(parents=True, exist_ok=True)
    datasets, evaluations, splits = {}, {}, {}
    inputs = {}
    started = perf_counter()
    for match in MATCHES:
        for path in map(Path, dfl_load.cache_paths(match)):
            inputs[str(path.relative_to(ROOT))] = digest(path)
        datasets[match], evaluations[match] = prepare_joint(match)
        splits[match] = internal_split(datasets[match])
        print(f"{match} 준비 | 학습 창 {len(splits[match][0])} | 내부검증 창 "
              f"{sum(map(len, splits[match][1]))} | 바깥 검증 프레임 {len(evaluations[match]['label']):,}", flush=True)
    preparation_s = perf_counter() - started
    fingerprint = dict(notebook_sha256=digest(NOTEBOOK), base_notebook_sha256=digest(original),
                       input_sha256=inputs, amp=AMP, device=str(DEVICE))
    results = []
    for held in MATCHES:
        fold_started = perf_counter()
        matches = [m for m in MATCHES if m != held]
        assert len(matches) == 6 and held not in matches
        print(f"{held} 바깥 평가 | 학습/내부검증 {', '.join(matches)}", flush=True)
        model, history, best_epoch, best_score = fit_joint(held, matches, datasets, splits)
        evaluation = evaluations[held]
        prob, ball, sampled = merge_predictions(model, evaluation)
        scores, viterbi, forced = score_joint(evaluation, prob)
        files = write_frames(held, evaluation, prob, sampled, viterbi, forced)
        ball_is_valid = (evaluation["label"] == 0) & np.isfinite(evaluation["ball_m"]).all(axis=-1)
        ade_m = float(np.linalg.norm(ball[ball_is_valid] - evaluation["ball_m"][ball_is_valid], axis=1).mean())
        result = dict(match=held, train_matches=matches, best_epoch=best_epoch, stopped_epoch=len(history),
                      best_inner_f1=best_score, ball_ade_in_play_m=ade_m, frames=len(prob),
                      fold_s=perf_counter() - fold_started,
                      scores=scores, training_s=sum(r["seconds"] for r in history),
                      file_sha256={str(path.relative_to(OUT)): digest(path) for path in files})
        results.append(result)
        print(f"{held} 완료 | epoch {best_epoch}/{len(history)} | 보정 후 프레임 F1 {scores[-1]['dead_f1']:.4f} "
              f"| 구간 F1 {scores[-1]['alive_spell_f1']:.4f} | 공 ADE {ade_m:.2f}m | 전체 {result['fold_s']:.1f}s", flush=True)

    per_match = pd.DataFrame([
        dict(match=fold["match"], best_epoch=fold["best_epoch"], stopped_epoch=fold["stopped_epoch"],
             ball_ade_in_play_m=fold["ball_ade_in_play_m"], training_s=fold["training_s"], fold_s=fold["fold_s"], **row)
        for fold in results for row in fold["scores"]
    ])
    assert len(per_match) == 21 and not per_match.duplicated(["match", "stage"]).any()
    assert set(per_match.match) == set(MATCHES) and per_match.dead_f1.between(0, 1).all()
    assert per_match.alive_spell_f1.between(0, 1).all()
    summary = (per_match.groupby("stage", sort=False)
               .agg(dead_frame_f1=("dead_f1", "mean"), dead_recall=("dead_recall", "mean"),
                    dead_precision=("dead_precision", "mean"), frame_acc=("frame_acc", "mean"),
                    alive_spell_f1=("alive_spell_f1", "mean"), ball_ade_in_play_m=("ball_ade_in_play_m", "mean"),
                    training_s=("training_s", "sum"), fold_s=("fold_s", "sum"), min_best_epoch=("best_epoch", "min"),
                    max_best_epoch=("best_epoch", "max")).reset_index())
    per_match.to_csv(OUT / "per_match.csv", index=False)
    summary.to_csv(OUT / "model_summary.csv", index=False)
    config = dict(width=WIDTH, heads=HEADS, max_epochs=EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA,
                  min_epochs=MIN_EPOCHS, inner_share=INNER_SHARE, state_weight=STATE_WEIGHT, batch=BATCH,
                  lr=LR, step=STEP, window=WINDOW, seed=SEED, model_parameters=parameters)
    names = ("per_match.csv", "model_summary.csv")
    save_json(dict(run_id=RUN_ID, fingerprint=fingerprint, config=config, matches=results,
                   preparation_s=preparation_s, code_commit=subprocess.check_output(
                       ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
                   versions={name: version(name) for name in ("torch", "pandas", "numpy", "pyarrow")},
                   report_sha256={name: digest(OUT / name) for name in names}), OUT / "run_manifest.json")
    save_json(dict(passed=True, held_matches=7, score_rows=len(per_match),
                   frames=sum(fold["frames"] for fold in results), smoke_test=True), OUT / "validation.json")
    print(summary.round(4).to_string(index=False), flush=True)
    return summary


parameters = check_joint()
print(f"device={DEVICE} | width={WIDTH} | heads={HEADS} | params={parameters:,} | AMP={AMP}", flush=True)
if os.environ.get("MTAH_TRANSPORTMER_MODE") == "benchmark":
    benchmark_joint()
else:
    summary = run_joint(parameters)